# DN-SC: run the GPT-2 study on a GPU and compare every number

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

What the notebook does, in order:

1. **Setup** — clones the repository (branch `review-fixes`, the reviewed code) and installs it with the library versions the published results used.
2. **Self-test** — the test suite, then the complete protocol on a tiny model *on the GPU*. If this passes, the code runs on this machine.
3. **Study** — the chosen profile with pretrained GPT-2. Finished runs are skipped, so after a disconnection you can simply run the cells again (keep `PERSIST_ON_DRIVE = True` so the runs survive).
4. **Verdict** — `COMPARISON.md` against the published pilot, and the report `TABLES.md`. Then a zip of everything to download.

**Which profile.** `"core"` (the default) is the confirmatory study of the paper (`docs/METHOD.md`, section 9): 88 stream runs on six seeds with the refinement decoder chosen on validation text in version 0.4 (beam 4, first round of 6 symbols), the decoder selection repeated as a check, and the real-decoding endpoint with its intervals (`tables/realdecode_hier.md`). Expect roughly 8–10 hours on a T4 (estimated from the CPU timings of version 0.4 and the T4 timings of the published runs), over more than one session. Because the decoder differs from the one the pilot used, `COMPARISON.md` will say that the two folders follow different protocols — that is expected; the results are in `TABLES.md`. `"published"` repeats the 37 pilot runs with their own decoder (already done: `results/gpt2_v0.3_colab_t4`, REPRODUCED). `"paper"` adds the sensitivity studies.

A CPU and a GPU do not produce bit-identical floating-point results, so agreement is judged against a tolerance (3 % on the headline numbers) and against the conclusions. Nothing in this notebook edits a result by hand.

In [ ]:
# ---- settings -------------------------------------------------------------------------
REPO    = "https://github.com/abraheemrashid99-sys/Wiley_paper.git"
BRANCH  = "review-fixes"   # the reviewed code (version 0.4.0); "main" still holds version 0.2
PROFILE = "core"           # "core": the confirmatory study, six seeds (about 8-10 h on a T4) | "published": the 37 pilot runs | "paper": core + sensitivity
REUSE_CODEBOOK = False     # True: start from the published codebook instead of training one (fewer cross-machine differences; retraining is the stricter test)
PERSIST_ON_DRIVE = True    # keep runs on Google Drive so a disconnected session can be resumed
RUN_SELFTEST = True        # the test suite and a tiny-model run on the GPU before the real study
DRY_RUN = False            # True: only print what would be run (for checking the notebook itself)

In [ ]:
import os, subprocess, sys, pathlib, shutil

def sh(command, check=True):
    print("$", command, flush=True)
    code = subprocess.run(command, shell=True).returncode
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")
    return code

# --- the repository
if pathlib.Path("Wiley_paper/scripts/reproduce.py").exists():
    os.chdir("Wiley_paper")
elif not pathlib.Path("scripts/reproduce.py").exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO} Wiley_paper")
    os.chdir("Wiley_paper")
sh("git fetch -q --depth 1 origin " + BRANCH + " && git checkout -q FETCH_HEAD", check=False)  # the latest commit of the branch
print("working directory:", os.getcwd())
print("code revision:", subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# --- install: the exact versions behind results/gpt2 for the libraries that shape the numbers;
#     torch stays the one Colab ships with (it carries the GPU build for this machine)
sh(f'{sys.executable} -m pip install -q "transformers==5.19.0" "tokenizers==0.23.3" "datasets==5.1.0" "sacrebleu==2.6.0" "pyzipper==0.4.0" "onnx>=1.15" "pytest>=8.0"')
sh(f'{sys.executable} -m pip install -q --no-deps -e .')

import torch, transformers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU - select a T4 under Runtime > Change runtime type")

In [ ]:
# --- where the runs live
RUNS = "runs"
if PERSIST_ON_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUNS = "/content/drive/MyDrive/dnsc_runs"
        os.makedirs(RUNS, exist_ok=True)
    except Exception as error:
        print("Drive not available, runs stay in this session:", error)
OUT, EXPORT = f"{RUNS}/gpt2_{PROFILE}", f"results_repro/gpt2_{PROFILE}"
EXTRA = (" --reuse-codebook" if REUSE_CODEBOOK else "") + (" --dry-run" if DRY_RUN else "")
print("runs:", OUT, "| exported results:", EXPORT, "| options:", EXTRA or "none")

## 1. Self-test

The unit and integration tests (tiny model, CPU), then the whole reproduction protocol on a tiny model on the selected device, with shortened streams. The numbers of the tiny model mean nothing; what matters is that every step runs here.

In [ ]:
if RUN_SELFTEST and not DRY_RUN:
    sh(f"{sys.executable} -m pytest -q -x")
    SHORT = "stream.tau_slow=14 stream.tau_med=7 stream.temporal_steps=28 pretrain.steps=100 pretrain.eval_every=50 eval.n_test=16"
    sh(f"{sys.executable} scripts/reproduce.py --config configs/smoke.yaml --out {RUNS}/selftest --export results_repro/selftest --profile published device={DEVICE} {SHORT}")
    print("SELF-TEST PASSED on", DEVICE)

## 2. Reproduction with pretrained GPT-2

The first cell builds GPT-2 (124M) from files hosted on GitHub (checksum verified). The second runs the study: one line per step, then a progress line every 50 steps of each run. If the session disconnects, run the notebook again; finished runs are skipped.

In [ ]:
if not pathlib.Path("models/gpt2/config.json").exists():
    sh(f"{sys.executable} scripts/fetch_gpt2.py --out models/gpt2")

In [ ]:
sh(f"{sys.executable} scripts/reproduce.py --config configs/gpt2.yaml --out {OUT} --export {EXPORT} --reference results/gpt2 --profile {PROFILE} device={DEVICE}{EXTRA}")

## 3. Verdict

For `"published"`: `REPRODUCED` means every headline number is within the tolerance and every conclusion holds; otherwise the report lists exactly what differs. For `"core"` and `"paper"` the decoder differs from the pilot's, so the comparison is between two protocols and is expected to differ; the results of the study are in `TABLES.md` (next cell) — first the headline decoded for real and `tables/realdecode_hier.md` with its intervals and seed-level tests.

In [ ]:
from IPython.display import Markdown, display
for name in ("COMPARISON.md", "TABLES.md"):
    report = pathlib.Path(EXPORT) / name
    if report.exists():
        display(Markdown(report.read_text(encoding="utf-8")))
    else:
        print("no report yet (dry run, or the study did not finish):", report)

In [ ]:
# everything the reproduction produced, as one zip to download (also kept on Drive when enabled)
if pathlib.Path(EXPORT).exists():
    archive = shutil.make_archive(f"dnsc_{PROFILE}_reproduction", "zip", EXPORT)
    if PERSIST_ON_DRIVE and RUNS.startswith("/content/drive"):
        shutil.copy2(archive, RUNS)
    print("archive:", archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception:
        pass